## Import libraries

In [1]:
from intertidal.extents import xr_cost_distance #load_connectivity_mask
from intertidal.io import load_data, extract_geobox

from dea_tools.spatial import xr_rasterize, xr_interpolate
from dea_tools.dask import create_local_dask_cluster
from odc.geo.geom import BoundingBox
from odc.geo.geobox import GeoBox
from odc.geo.gridspec import GridSpec
from odc.geo.types import xy_
from odc.algo import mask_cleanup
import odc.geo.xr

import datacube
import os
import json
import numpy as np
import geopandas as gpd
import xarray as xr
from skimage import graph
import gc

In [2]:
pip install pyogrio

  Using cached pyogrio-0.10.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
Using cached pyogrio-0.10.0-cp310-cp310-manylinux_2_28_x86_64.whl (23.9 MB)
Note: you may need to restart the kernel to use updated packages.


## Connect to the datacube

In [2]:
# Create local dask cluster to improve data load time
client = create_local_dask_cluster(return_client=True)

# Connect to datacube to load data
dc = datacube.Datacube(app="connectivity_testing")

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/claire.phillips@ga.gov.au/proxy/8787/status,
Dashboard: /user/claire.phillips@ga.gov.au/proxy/8787/status,Workers: 1
Total threads: 62,Total memory: 477.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37283,Workers: 1
Dashboard: /user/claire.phillips@ga.gov.au/proxy/8787/status,Total threads: 62
Started: Just now,Total memory: 477.21 GiB
Comm: tcp://127.0.0.1:39983,Total threads: 62
Dashboard: /user/claire.phillips@ga.gov.au/proxy/42989/status,Memory: 477.21 GiB
Nanny: tcp://127.0.0.1:38823,


## Load CEM tile list

In [3]:
# # Load the GeoJSON file
# with open('/gdata1/data/albers_grids/coastal/CEM grid/ga_summary_grid_c3_32km_cem.geojson', 'r') as f:
#     geojson_data = json.load(f)

# Load the GeoJSON file
with open('/gdata1/projects/coastal/cem/grids/ga_summary_grid_96km_coastal_CEM_grid.geojson', 'r') as f:
    geojson_data = json.load(f)

# geojson_data

# Extract the 'id' for each feature
ids = [feature.get('properties', None)['region_code'] for feature in geojson_data.get('features', [])]

# Print or store the extracted ids
# print(ids)

#### Remove already processed tiles from /gdata1/projects/coastal/intertidal/connectivity_mask_96km/

In [8]:
## Robbi's cell

# Identify already processed tiles


#Import list of file names

directory = '/gdata1/projects/coastal/intertidal/connectivity_mask_96km/nomangroves/'
files=os.listdir(path=directory)
#Reduce file names in list to tile names only
duplicated_tiles = [file.split('_')[2] for file in files if os.path.isfile(os.path.join(directory, file))]
#Remove duplicates
processed_tiles = list(set(duplicated_tiles))

print(len(duplicated_tiles), len(processed_tiles))

# Compare CEM ids and processed_tiles
print (len(ids), len(processed_tiles))

# remove processed tiles from ids
unprocessed_tiles = [item for item in ids if item not in processed_tiles]

unprocessed_tiles = [item for item in unprocessed_tiles if item not in tiles_error]
print(len(unprocessed_tiles))

24 6
325 6
319


In [12]:
## Claire's cell

# Identify already processed tiles


#Import list of file names

directory = '/gdata1/projects/coastal/intertidal/connectivity_mask_96km/mangroves/'
files=os.listdir(path=directory)
#Reduce file names in list to tile names only
duplicated_tiles = [file.split('_')[2] for file in files if os.path.isfile(os.path.join(directory, file))]
#Remove duplicates
processed_tiles = list(set(duplicated_tiles))

print(len(duplicated_tiles), len(processed_tiles))

# Compare CEM ids and processed_tiles
print (len(ids), len(processed_tiles))

# remove processed tiles from ids
unprocessed_tiles = [item for item in ids if item not in processed_tiles]

unprocessed_tiles = [item for item in unprocessed_tiles if item not in tiles_error]
print(len(unprocessed_tiles))

12 6
325 6
319


### Define functions to generate connectivity masks

In [5]:
def load_connectivity_mask(
    dc,
    geobox,
    product="ga_srtm_dem1sv1_0",
    elevation_band="dem_h",
    resampling="bilinear",
    buffer=20000,
    preprocess=None,
    max_threshold=100,
    add_mangroves=False,
    correct_hat=False,
    mask_filters=[("dilation", 3)],
    **cost_distance_kwargs,
):
    """
    Generates a mask based on connectivity to ocean pixels, using least-
    cost distance weighted by elevation. By incorporating elevation,
    this mask will extend inland further in areas of low lying elevation
    and less far inland in areas of steep terrain.

    Parameters
    ----------
    dc : Datacube
        A Datacube instance for loading data.
    geobox : ndarray
        The GeoBox defining the pixel grid to load data into (e.g.
        resolution, extents, CRS).
    product : str, optional
        The name of the DEM product to load from the datacube.
        Defaults to "ga_srtm_dem1sv1_0".
    elevation_band : str, optional
        The name of the band containing elevation data. Defaults to
        "height_depth".
    resampling : str, optional
        The resampling method to use, by default "bilinear".
    buffer : int, optional
        The distance by which to buffer the input GeoBox to reduce edge
        effects. This buffer will eventually be removed and clipped back
        to the original GeoBox extent. Defaults to 20,000 metres.
    preprocess : function, optional
        An optional lambda function that can be applied to the elevation
        data prior to connecitivity analysis. Regardless of the outputs
        of this function, the resulting data will always be clipped
        between 0 and inf so that it is suitable for analysis. Defaults
        to None.
    max_threshold: int, optional
        Value used to threshold the resulting cost distance to produce
        a mask.
    add_mangroves : bool, optional
        Whether to use the extent of mangroves from Global Mangrove Watch
        as additional starting points for the connectivity analysis.
        Defaults to False.
    correct_hat : bool, optional
        Whether to apply a Highest Astronomical Tide correction, to make
        costs based on height above HAT vs height above MSL. Defaults to
        False.
    mask_filters : list of tuples, optional
        An optional list of morphological processing steps to pass to
        the `mask_cleanup` function. The default is `[("dilation", 3)]`,
        which will dilate True pixels by a radius of 3 pixels.
    **cost_distance_kwargs :
        Optional keyword arguments to pass to the ``xr_cost_distance``
        cost-distance function.

    Returns
    -------
    costdist_mask : xarray.DataArray
        An output boolean mask, where True represent pixels located in
        close cost-distance proximity to the ocean.
    costdist_da : xarray.DataArray
        The output cost-distance array, reflecting distance from the
        ocean weighted by elevation.
    """

    # Buffer input geobox and reduce resolution to ensure that the
    # connectivity analysis is less affected by edge effects
    print("Loading SRTM data at native 30 m resolution")
    geobox_buffered = GeoBox.from_bbox(
        geobox.buffered(xbuff=buffer, ybuff=buffer).boundingbox,
        resolution=30,
        tight=True,
    )

    # Load DEM data
    dem_da = dc.load(
        product="ga_srtm_dem1sv1_0",
        measurements=[elevation_band],
        resampling="bilinear",
        like=geobox_buffered,
    ).squeeze()[elevation_band]

    # Identify starting points (ocean nodata points)
    if add_mangroves:
        print("Adding GMW mangroves to starting points")
        try:
            gmw_da = load_gmw_mask(dem_da)
            starts_da = (dem_da == dem_da.nodata) | gmw_da
        except:
            starts_da = dem_da == dem_da.nodata
    else:
        starts_da = dem_da == dem_da.nodata

    # Raise error if no valid starting points
    if not starts_da.any():
        raise Exception("No valid starting points found for tile, likely due to being located too far inland")

    # Apply a Highest Astronomical Tide correction, to make
    # costs based on height above HAT vs height above MSL
    if correct_hat:
        print("Applying HAT correction")
        hat_correction = load_hat(dem_da)
        dem_da = dem_da - hat_correction.data

    # Calculate cost surface, optionally using custom preprocess
    # function (negative values are not allowed, so negative
    # nodata values are resolved by clipping values to between
    # 0 and infinity)
    if preprocess is not None:
        print("Using custom pre-process function")
        costs_da = preprocess(dem_da).clip(0, np.inf)
    else:
        costs_da = dem_da.clip(0, np.inf)

    # Run cost distance surface
    print("Running cost distance calculation")
    costdist_da = xr_cost_distance(
        cost_da=costs_da,
        starts_da=starts_da,
        **cost_distance_kwargs,
    )        

    # Reproject back to original geobox extents and resolution
    print("Reprojecting back to original GeoBox resolution")
    costdist_da = costdist_da.odc.reproject(how=geobox, resampling="bilinear")

    # Apply threshold
    costdist_mask = costdist_da < max_threshold

    # If requested, apply cleanup
    if mask_filters is not None:
        costdist_mask = mask_cleanup(costdist_mask, mask_filters=mask_filters)

    return costdist_mask, costdist_da


def load_gmw_mask(ds, gmw_path="/gdata1/data/mangroves/gmw_v3_2020_vec_aus.geojson"):
    """
    Experiment with loading GMW data to use as additional
    starting points in connectivity analysis.
    """
    gmw_gdf = gpd.read_file(
        gmw_path, bbox=ds.odc.geobox.boundingbox.to_crs("EPSG:4326")
    )
    gmw_da = xr_rasterize(gmw_gdf, ds)
    return gmw_da


def load_hat(
    ds,
    hat_path="/gdata1/data/tide_datums/HAT_MLP_Regression.gpkg",
    layer="HAT_MLP_Regression",
    hat_col="HAT",
    interp_method="idw",
    interp_p=1,
    interp_k=10,
    interp_factor=100,
):
    """
    Experiment with interpolating CSIRO HAT data to use
    as a correction to elevation values.

    Branson, Paul (2023): Coastal carbon - Australia's blue forest
    future - Water Levels. v1. CSIRO. Data Collection.
    https://doi.org/10.25919/6672-jx11
    """
    # Load CSIRO HAT data
    hat = gpd.read_file(
        hat_path,
        layer=layer,
        engine="pyogrio",
    ).drop(["x", "y"], axis=1)

    # Interpolate into extent of data
    hat_correction = xr_interpolate(
        ds=ds,
        gdf=hat[[hat_col, "geometry"]],
        method=interp_method,
        p=interp_p,
        k=interp_k,
        factor=interp_factor,
    ).HAT

    return hat_correction

## Calculate connectivity for CEM tileset 

In [14]:
tiles_error=[]

In [21]:
# ## Robbi's cell

# # Loop through 96km tiles to generate HAT/Mangrove/Ocean connectivity masks 11/10/2024
# buffer = 48000

# for study_area in unprocessed_tiles:

#     print(f"Processing {study_area}")
    
#     geobox = extract_geobox(study_area=study_area, resolution=30, tile_width=96000)
#     try:
#         # Calculate ocean connectivity array
#         _, costdist_da = load_connectivity_mask(
#             dc, 
#             geobox,
#             max_threshold=100,
#             buffer=buffer,
#             add_mangroves=False,
#             correct_hat=False,
#             mask_filters=None,
#         )

#         costdist_da.clip(0, 32767).astype("int16").odc.write_cog(
#             f"/gdata1/projects/coastal/intertidal/connectivity_mask_96km/nomangroves/connectivity_dist_{study_area}_nomangroves_nohat.tif", overwrite=True, nodata=-999
#         )

#         # Calculate ocean-hat connectivity array
#         _, costdist_da = load_connectivity_mask(
#             dc, 
#             geobox,
#             max_threshold=100,
#             buffer=buffer,
#             add_mangroves=False,
#             correct_hat=True,
#             mask_filters=None,
#         )

#         costdist_da.clip(0, 32767).astype("int16").odc.write_cog(
#             f"/gdata1/projects/coastal/intertidal/connectivity_mask_96km/nomangroves/connectivity_dist_{study_area}_nomangroves_hat.tif", overwrite=True, nodata=-999
#         )

#         gc.collect()
        
#     except Exception as e:
#         print(f'An error occurred in {study_area}: {e}')
#         tiles_error.append(study_area)
#         continue

In [15]:
## Claire's cell

# Loop through 96km tiles to generate HAT/Mangrove/Ocean connectivity masks 11/10/2024
buffer = 48000

for study_area in unprocessed_tiles:

    print(f"Processing {study_area}")
    
    geobox = extract_geobox(study_area=study_area, resolution=30, tile_width=96000)
    try:

        # Calculate ocean-mangrove connectivity array
        _, costdist_da = load_connectivity_mask(
            dc, 
            geobox,
            max_threshold=100,
            buffer=buffer,
            add_mangroves=True,
            correct_hat=False,
            mask_filters=None,
        )

        costdist_da.clip(0, 32767).astype("int16").odc.write_cog(
            f"/gdata1/projects/coastal/intertidal/connectivity_mask_96km/mangroves/connectivity_dist_{study_area}_mangroves_nohat.tif", overwrite=True, nodata=-999
        )

        # Calculate ocean-mangrove-hat connectivity array
        _, costdist_da = load_connectivity_mask(
            dc, 
            geobox,
            max_threshold=100,
            buffer=buffer,
            add_mangroves=True,
            correct_hat=True,
            mask_filters=None,
        )

        costdist_da.clip(0, 32767).astype("int16").odc.write_cog(
            f"/gdata1/projects/coastal/intertidal/connectivity_mask_96km/mangroves/connectivity_dist_{study_area}_mangroves_hat.tif", overwrite=True, nodata=-999
        )
        
        gc.collect()
        
    except Exception as e:
        print(f'An error occurred in {study_area}: {e}')
        tiles_error.append(study_area)
        continue

Processing x57y23
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation


2024-10-11 05:35:00,336 - distributed.utils_perf - WARNING - full garbage collections took 14% CPU time recently (threshold: 10%)


Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction
Running cost distance calculation


2024-10-11 05:38:05,869 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2024-10-11 05:38:13,665 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)


Reprojecting back to original GeoBox resolution
Processing x58y23
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x60y23
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x57y24
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at

...

Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x36y58
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction


/tmp/ipykernel_312/1671559558.py:184: UserWarning: The supplied `gdf` does not overlap spatially with `ds`.
  hat_correction = xr_interpolate(


Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x43y58
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x44y58
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x45y58
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoB

/tmp/ipykernel_312/1671559558.py:184: UserWarning: The supplied `gdf` does not overlap spatially with `ds`.
  hat_correction = xr_interpolate(


Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x43y59
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x44y59
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Applying HAT correction
Running cost distance calculation
Reprojecting back to original GeoBox resolution
Processing x45y59
Loading SRTM data at native 30 m resolution
Adding GMW mangroves to starting points
Running cost distance calculation
Reprojecting back to original GeoB

In [9]:
# # Loop through 32km tiles to generate Mangrove/Ocean connectivity masks 27/09/2024

# for study_area in unprocessed_tiles:
#     print(f"Processing {study_area}")
#     geobox = extract_geobox(study_area=study_area)
#     try:
#         # Calculate ocean connectivity array
#         _, costdist_da = load_connectivity_mask(dc, geobox)

#         costdist_da.clip(0, 32767).astype("int16").odc.write_cog(
#             f"/gdata1/projects/coastal/intertidal/connectivity_mask/connectivity_dist_{study_area}_nomangroves.tif", overwrite=True, nodata=-999
#         )

#         # Calculate ocean-mangrove connectivity array
#         _, costdist_da = load_connectivity_mask(dc, geobox, add_mangroves=True)

#         costdist_da.clip(0, 32767).astype("int16").odc.write_cog(
#             f"/gdata1/projects/coastal/intertidal/connectivity_mask/connectivity_dist_{study_area}_mangroves.tif", overwrite=True, nodata=-999
#         )
#     except Exception as e:
#         print(f'An error occurred in {study_area}: {e}')
#         tiles_error.append(study_area)
#         continue

## COG the outputs

In [16]:
## Claire's cell

input_path = "/gdata1/projects/coastal/intertidal/connectivity_mask_96km/mangroves/*dist*_mangroves_nohat.tif"
output_path = "/gdata1/projects/coastal/intertidal/continental_connectivity_dist_mangroves_nohat.tif" 

!gdalwarp $input_path $output_path -b 1 -overwrite -multi -wm 80% -co NUM_THREADS=ALL_CPUS -of COG -co COMPRESS=DEFLATE -co PREDICTOR=YES

0...10...20...30...40...50...60...70...80...90...100 - done.


In [ ]:
## Robbi's cell

input_path = "/gdata1/projects/coastal/intertidal/connectivity_mask_96km/nomangroves/*dist*_nomangroves_nohat.tif"
output_path = "/gdata1/projects/coastal/intertidal/continental_connectivity_dist_nomangroves_nohat.tif" 

!gdalwarp $input_path $output_path -b 1 -overwrite -multi -wm 80% -co NUM_THREADS=ALL_CPUS -of COG -co COMPRESS=DEFLATE -co PREDICTOR=YES

In [ ]:
## Robbi's cell

input_path = "/gdata1/projects/coastal/intertidal/connectivity_mask_96km/nomangroves/*dist*_nomangroves_hat.tif"
output_path = "/gdata1/projects/coastal/intertidal/continental_connectivity_dist_nomangroves_hat.tif" 

!gdalwarp $input_path $output_path -b 1 -overwrite -multi -wm 80% -co NUM_THREADS=ALL_CPUS -of COG -co COMPRESS=DEFLATE -co PREDICTOR=YES

In [17]:
## Claire's cell

input_path = "/gdata1/projects/coastal/intertidal/connectivity_mask_96km/mangroves/*dist*_mangroves_hat.tif"
output_path = "/gdata1/projects/coastal/intertidal/continental_connectivity_dist_mangroves_hat.tif" 

!gdalwarp $input_path $output_path -b 1 -overwrite -multi -wm 80% -co NUM_THREADS=ALL_CPUS -of COG -co COMPRESS=DEFLATE -co PREDICTOR=YES

0...10...20...30...40...50...60...70...80...90...100 - done.


In [18]:
print(tiles_error)

[]


In [19]:
## Claire's cell

with open("/gdata1/projects/coastal/intertidal/failed_mangrove_96km_CEM_tiles.txt","w") as file:
    for item in tiles_error:
        file.write(f"{item}/n")

In [ ]:
## Robbi's cell

with open("/gdata1/projects/coastal/intertidal/failed_nomangrove_96km_CEM_tiles.txt","w") as file:
    for item in tiles_error:
        file.write(f"{item}/n")

In [20]:
len(tiles_error)

0